# Step 2: Embeddings

**Goal:** turn each ID into a short list of numbers (a **vector**), so that tokens with similar meaning get similar numbers.

The **embedding table** has one row per token (53 rows) and 8 columns (our vector size). To embed a token, read its row. That's all.

In [ ]:
# Make the `toddler` package (repo root) importable from this notebook
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.exercises import check, solution

np.set_printoptions(formatter={"float_kind": lambda x: f"{x:g}"})
pd.set_option("display.float_format", "{:g}".format)

tok = BPETokenizer.train(CORPUS)
print(len(tok.vocab), "tokens in the vocabulary")

## 2.1 At birth: random numbers

Every number in the table starts random. Training (Step 7) will change them. For now, the column names are only labels.

In [ ]:
COLUMNS = ["me", "be", "want", "stop", "feel", "food", "hug", "sleep"]

def random_table(vocab_size, dim, seed=0):
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, 0.5, size=(vocab_size, dim)).round(2) + 0.0  # + 0.0 turns -0.0 into 0.0

table = random_table(len(tok.vocab), len(COLUMNS))
print("shape:", table.shape, "=", table.size, "numbers")
pd.DataFrame(table, index=tok.vocab, columns=COLUMNS).loc[["_I", "_hungry", "_milk", "_hun"]]

## 2.2 After training (cleaned up)

We jump ahead and write in the cleaned-up values a trained toddler **could** end up with. Real learned values look like random decimals, and their columns have no names. We use clean 0/1 values so we can follow the math by hand.

- The first 4 columns answer "which small word am I?"
- `feel` answers "is it a feeling?"
- `food`, `hug` and `sleep` answer "what is it about?" A feeling and the thing it wants share a topic.

Only the 10 tokens our sentences use get these values. The 43 unused rows (`_hun`, `h`, …) stay random.

In [ ]:
CLEAN_ROWS = {
    "_I":      [1, 0, 0, 0, 0, 0, 0, 0],
    "_am":     [0, 1, 0, 0, 0, 0, 0, 0],
    "_want":   [0, 0, 1, 0, 0, 0, 0, 0],
    "_.":      [0, 0, 0, 1, 0, 0, 0, 0],
    "_hungry": [0, 0, 0, 0, 1, 1, 0, 0],
    "_sad":    [0, 0, 0, 0, 1, 0, 1, 0],
    "_tired":  [0, 0, 0, 0, 1, 0, 0, 1],
    "_milk":   [0, 0, 0, 0, 0, 1, 0, 0],
    "_mommy":  [0, 0, 0, 0, 0, 0, 1, 0],
    "_bed":    [0, 0, 0, 0, 0, 0, 0, 1],
}
for token, values in CLEAN_ROWS.items():
    table[tok.token_to_id[token]] = values

def row(token):
    return table[tok.token_to_id[token]]

pd.DataFrame(table, index=tok.vocab, columns=COLUMNS).loc[list(CLEAN_ROWS) + ["_hun"]]

## 2.3 Lookup, step by step

Each ID is a row number: read that row.

In [ ]:
sentence = "I am hungry . I want milk ."
ids = tok.encode(sentence)
for position, token_id in enumerate(ids, start=1):
    print(f"pos {position}: {tok.vocab[token_id]:<8} → ID {token_id:>2} → row {token_id:>2} = {table[token_id]}")

Stacked together, the sentence becomes a grid: **this grid is what goes into the transformer**. It has 8 rows because the sentence has 8 tokens, and 8 columns because the vector size is 8 (the same number only by chance).

In [ ]:
grid = table[ids]  # NumPy reads all the rows at once
pd.DataFrame(grid, index=[f"{p} {tok.vocab[i]}" for p, i in enumerate(ids, start=1)], columns=COLUMNS)

### Lookup = one-hot × table

A one-hot vector is 53 zeros with a single 1 at the token's ID. Multiplying it by the table keeps only that row. So the embedding table is an ordinary weight matrix, and training can adjust it like any other weights.

In [ ]:
one_hot = np.zeros(len(tok.vocab))
one_hot[tok.token_to_id["_hungry"]] = 1
print("one-hot(_hungry):", one_hot.astype(int))
print("one-hot × table :", one_hot @ table)
print("row 36          :", table[36])
print("same?           ", np.array_equal(one_hot @ table, table[36]))

## 2.4 Measuring "close": the dot product

Multiply two vectors column by column, then **add** the results. Shared topics give a big number; nothing in common gives 0. Attention (Step 4) is built on this.

In [ ]:
def show_dot(a, b):
    va, vb = row(a), row(b)
    print(f"{'':<10}" + "".join(f"{c:>6}" for c in COLUMNS))
    print(f"{a:<10}" + "".join(f"{x:>6g}" for x in va))
    print(f"{b:<10}" + "".join(f"{x:>6g}" for x in vb))
    print(f"{'multiply':<10}" + "".join(f"{x:>6g}" for x in va * vb) + f"   → sum = {va @ vb:g}\n")

show_dot("_hungry", "_sad")
show_dot("_hungry", "_milk")
show_dot("_hungry", "_mommy")

All the pairs at once: `vectors @ vectors.T` computes every dot product in one go.

In [ ]:
import matplotlib.pyplot as plt

content = ["_hungry", "_sad", "_tired", "_milk", "_mommy", "_bed"]
vectors = np.array([row(t) for t in content])
similarity = vectors @ vectors.T

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(similarity, cmap="Blues")
ax.set_xticks(range(len(content)), content, rotation=45)
ax.set_yticks(range(len(content)), content)
for i in range(len(content)):
    for j in range(len(content)):
        ax.text(j, i, f"{similarity[i, j]:g}", ha="center", va="center")
ax.set_title("dot products")
plt.show()

- The 3 feelings are close to each other.
- Each feeling is close to **its own** thing.
- milk, mommy and bed are **not** close to each other, although they all come after `want`. 8 columns are too few to store everything; exercise 2.4 fixes this with a 9th column.

## 2.5 Adding and subtracting meaning

Column by column, like a sum on paper. To name the result, we pick the token at the smallest distance from it.

In [ ]:
result = row("_milk") - row("_hungry") + row("_sad")

print(f"{'':<10}" + "".join(f"{c:>6}" for c in COLUMNS))
for label, vector in [("  _milk", row("_milk")), ("- _hungry", row("_hungry")), ("+ _sad", row("_sad")), ("= result", result)]:
    print(f"{label:<10}" + "".join(f"{x:>6g}" for x in vector))

closest = min(CLEAN_ROWS, key=lambda t: np.linalg.norm(result - row(t)))
print("\nclosest token:", closest)

## 2.6 Unused rows stay random

Tokens like `_hun` never appear in the training text, so training never touches their rows, and their dot products mean nothing. In real models, such rarely seen tokens cause strange "glitch token" behavior.

In [ ]:
print("_hun row:", row("_hun"))
for t in ["_hungry", "_milk", "_I"]:
    print(f"_hun · {t:<8} = {row('_hun') @ row(t):+.2f}")

## 2.7 What is still missing

Rows 1 and 5 of the grid are both `_I`:

In [ ]:
print("pos 1:", grid[0])
print("pos 5:", grid[4])
print("identical?", np.array_equal(grid[0], grid[4]))

Same input, same output: the model would make the same guess at both places. But the first `_I` is followed by `_am`, and the second by `_want`. Fix: **positions** (Step 3).

And row 6 (`_want`) only knows "want": nothing in it says the child is hungry. Fix: **attention** (Step 4).

## The package version

`toddler/embeddings.py` builds the same table. Later steps use it.

In [ ]:
from toddler import embeddings

assert np.array_equal(embeddings.toddler_table(tok), table)
print("toddler/embeddings.py builds the same table")

## Your turn

### 2.1 Compute `_tired · _bed` by hand (multiply column by column, then add).

In [ ]:
my_answer = None  # one number
check("2.1", my_answer)

### 2.2 Which token is `_bed - _tired + _hungry`?

In [ ]:
my_answer = None  # a token, e.g. "_sad"
check("2.2", my_answer)

### 2.3 Write the dot product yourself, with a loop (no `@`, no `np.dot`).

In [ ]:
def my_dot(a, b):
    total = 0
    # your code here
    return None

check("2.3", my_dot)

### 2.4 Add a 9th column, `thing`, so that milk, mommy and bed become close to each other

Put a 1 in the `thing` column of the right rows, and leave the other rows at 0.

In [ ]:
table9 = np.hstack([table, np.zeros((len(tok.vocab), 1))])
# your code here (tip: tok.token_to_id["_milk"] is the row of _milk; column 8 is the new one)

check("2.4", table9)

In [ ]:
# Run this after 2.4: the new dot products
vectors9 = np.array([table9[tok.token_to_id[t]] for t in content])
pd.DataFrame(vectors9 @ vectors9.T, index=content, columns=content)

## Summary

- The embedding table has one row per token. Embedding = reading the row of each ID.
- The numbers start random and are learned. Tokens used in similar ways end up with similar rows.
- The **dot product** measures how close two vectors are. Attention is built on it.
- Meaning can be added and subtracted: milk − hungry + sad = mommy.
- Each row only knows its own token: no position, no context. Next: **Step 3, positions**.